# Урок 28 — Теорія Графів, BFS/DFS, Dijkstra та Neo4j

**Модуль 3 · Python Advanced · Автор: Viktor Nikoriak**

---

## Про що цей урок?

Традиційно ми вивчаємо дані через прості структури:

- масиви → послідовності
- дерева → ієрархії
- алгоритми пошуку і сортування → оптимізація операцій над цими структурами

І це важливий фундамент.

Але є один момент:

> ці моделі добре працюють лише тоді, коли світ можна впорядкувати.

У реальності ж більшість систем виглядають інакше.

Вони не є ні списками, ні деревами.
Вони є мережею взаємозв’язків.

- міста і дороги
- користувачі і соціальні зв’язки
- транзакції і фінансові потоки
- водні системи і річкові басейни

Це вже не ієрархія.
Це — граф.

І тут відбувається зсув мислення:

> ми більше не питаємо “де лежить елемент?”
> ми питаємо “як елементи пов’язані?”

Саме тому графи — це не просто ще одна структура даних.

Це інший спосіб бачити систему.

У цьому уроці ми перейдемо:
- від лінійного мислення → до мережевого
- від обробки даних → до навігації по зв’язках
- від SQL JOIN → до Graph Traversal

І зрозуміємо, як алгоритми BFS, DFS і Dijkstra працюють у світі,
де головна цінність — це зв’язки.
---

## Структура уроку

| Частина | Тема |
|---------|------|
| 1 | Що таке граф? G = (V, E) |
| 2 | Типи ребер: направлені, ненаправлені, зважені |
| 3 | Зберігання графу в пам'яті: матриця vs. список суміжності |
| 4 | Дерева vs. Графи — ієрархія проти мережі |
| 5 | BFS — обхід у ширину |
| 6 | DFS — обхід у глибину |
| 7 | Dijkstra — найдешевший маршрут |
| 8 | Bellman-Ford — від'ємні ваги |
| 9 | SQL JOIN vs. Graph Traversal |
| 10 | Архітектура Neo4j — Index-Free Adjacency |
| 11 | Cypher — мова запитів |
| 12 | Реальні застосування |

---


## Частина 1: Що таке граф? G = (V, E)

### Формальне визначення

Граф $G$ — це математична структура, визначена як **впорядкована пара**:

$$G = (V, E)$$

де:
- **$V$ (Vertices / Nodes / Вузли)** — множина сутностей. Вузол може представляти:
  людину, місто, маршрутизатор, банківський рахунок, DNS-запис, молекулу, зону таксі.
- **$E$ (Edges / Links / Ребра)** — множина зв'язків між вузлами.
  Ребро перетворює розрізнений список сутностей на **мережу**.

Складність алгоритмів графів вимірюється через $|V|$ (кількість вузлів) та $|E|$ (кількість ребер).

### Чому граф, а не таблиця?

Традиційні структури даних описують **ізольовані об'єкти**:
- Масив: `[5, 2, 8, 1]` — числа без контексту
- Таблиця: рядок = запис, колонка = атрибут

Граф моделює **зв'язки** — ту інформацію, яка **живе між** об'єктами:
- Хто кого знає?
- Яка дорога з'єднує міста?
- Чи веде трансакція від шахрая до жертви через 5 кроків?

> **Ключова ідея архітектора:** Граф зберігає реальність такою, якою вона є —
> не спотворюючи її у плоскі таблиці і не втрачаючи зв'язки між рядками.


In [ ]:
# ── Частина 1: базовий граф у Python ──────────────────────────────────────
# Найпростіша реалізація — словник списків (adjacency list)
# Кожен ключ = вузол, значення = список сусідів

# Граф зон NYC Taxi — неспрямований, незважений
# Ребро означає: між цими двома зонами є хоча б одна поїздка
taxi_graph_simple = {
    132: [138, 236, 11],   # Midtown -> JFK, Upper West Side, Brooklyn Heights
    138: [132, 3],          # JFK -> Midtown, Bronx
    236: [132, 92],         # Upper West Side -> Midtown, Jackson Heights
    11:  [132, 92],         # Brooklyn Heights -> Midtown, Jackson Heights
    92:  [236, 11, 3],      # Jackson Heights -> UWS, Brooklyn, Bronx
    3:   [138, 92],         # Bronx -> JFK, Jackson Heights
}

# Вузли (V) та ребра (E)
V = set(taxi_graph_simple.keys())
E = set()
for node, neighbors in taxi_graph_simple.items():
    for neighbor in neighbors:
        # frozenset щоб не дублювати ребра (A,B) і (B,A)
        E.add(frozenset([node, neighbor]))

print(f"Вузлів |V| = {len(V)}: {sorted(V)}")
print(f"Ребер  |E| = {len(E)}: {[sorted(e) for e in E]}")

## Частина 2: Типи ребер — направлені, ненаправлені, зважені

### 2.1 Неспрямований граф (Undirected Graph)

Ребро `{u, v}` — **симетричне**. Якщо Аліса друг Боба → Боб друг Аліси.

Приклади:
- Facebook-дружба (обидві сторони погоджуються)
- Двосмугова дорога між містами
- Фізичний кабель між двома серверами

### 2.2 Спрямований граф (Directed Graph / Digraph)

Ребро `(u, v)` — **одностороннє**, вказує від `u` до `v`.
Факт `(u→v)` **не означає** існування `(v→u)`.

Приклади:
- Twitter-підписки: я підписаний на Маска, але він не підписаний на мене
- Залежності між пакетами: `requests` залежить від `urllib3`, а не навпаки
- Транзакція: гроші йдуть від рахунку A до рахунку B

**In-degree / Out-degree:**
- `in-degree(v)` — кількість ребер, які **входять** у v (скільки людей підписані на вас)
- `out-degree(v)` — кількість ребер, які **виходять** з v (скільки людей підписані ВИ)

### 2.3 Зважений граф (Weighted Graph)

Кожне ребро несе числовий атрибут — **вагу** (weight / cost):
- **GPS**: вага = відстань у км або час у хвилинах
- **Фінанси**: вага = вартість транзакції або відсоткова ставка
- **Соціальні мережі**: вага = частота комунікації
- **NYC Taxi**: вага = середня вартість поїздки між двома зонами

Без ваг — BFS знаходить **найменшу кількість переходів**.  
З вагами — Dijkstra/A* знаходять **найдешевший/найшвидший маршрут**.


In [ ]:
# ── Частина 2: зважений спрямований граф ──────────────────────────────────
# Структура: {вузол: [(сусід, вага), ...]}
# Вага = середня вартість поїздки між зонами (USD)

taxi_graph_weighted = {
    132: [(138, 45.3), (236, 12.1), (11, 24.7)],
    138: [(132, 44.8), (3, 38.2)],
    236: [(132, 11.9), (92, 28.5)],
    11:  [(132, 23.9), (92, 15.4)],
    92:  [(236, 27.8), (11, 16.1), (3, 19.8)],
    3:   [(138, 37.6), (92, 20.1)],
}

# Перевірка: вивести всі ребра з вагами
print("Зважений граф зон NYC Taxi:")
print(f"{'Від':>6} → {'До':>6}  {'Вартість (USD)':>14}")
print("-" * 35)
for source, edges in taxi_graph_weighted.items():
    for dest, cost in edges:
        print(f"Zone {source:>3} → Zone {dest:>3}  ${cost:>12.1f}")

## Частина 3: Зберігання графу — матриця vs. список суміжності

Щоб реалізувати алгоритми, граф потрібно перекласти у структуру даних.
Вибір структури визначає складність алгоритмів.

### 3.1 Матриця суміжності (Adjacency Matrix)

Двовимірний масив розміром $N \times N$, де $N = |V|$.
Якщо ребро `(i, j)` існує → `matrix[i][j] = 1` або `= вага`.

| | A | B | C |
|--|--|--|--|
| **A** | 0 | 1 | 1 |
| **B** | 1 | 0 | 0 |
| **C** | 1 | 0 | 0 |

**Плюси:**
- Перевірка `edge(i, j)` — O(1)
- Ідеально інтегрується з матричними операціями (лінійна алгебра)

**Мінуси:**
- Пам'ять: $O(|V|^2)$ — якщо 1M користувачів = **1 трильйон комірок**
- Реальні мережі **розріджені** (sparse): більшість комірок = 0
- Непрактична для великих графів

### 3.2 Список суміжності (Adjacency List)

Словник або масив, де кожен вузол зберігає **локальний список** своїх сусідів.

**Плюси:**
- Пам'ять: $O(|V| + |E|)$ — тільки реальні ребра займають місце
- Стандарт для промислових графів
- Прямий аналог фізичної архітектури Neo4j (index-free adjacency)

**Мінуси:**
- Перевірка `edge(i, j)` — O(degree) замість O(1)

> **Правило архітектора:** для реальних мереж (соціальні, транспортні, фінансові)
> завжди обирайте **adjacency list** — мережі завжди розріджені.


In [ ]:
# ── Частина 3: матриця vs. список суміжності — порівняння пам'яті ─────────

import sys
from collections import defaultdict

# Симулюємо невеликий граф з 10 вузлів і 15 ребер (sparse)
N = 10
edges_list = [
    (0,1), (0,3), (1,2), (2,4), (3,5),
    (4,6), (5,7), (6,8), (7,9), (8,9),
    (1,5), (2,6), (3,7), (4,8), (0,9)
]

# Матриця суміжності: O(N^2)
matrix = [[0] * N for _ in range(N)]
for u, v in edges_list:
    matrix[u][v] = 1
    matrix[v][u] = 1

# Список суміжності: O(V + E)
adj_list = defaultdict(list)
for u, v in edges_list:
    adj_list[u].append(v)
    adj_list[v].append(u)

matrix_size = sys.getsizeof(matrix) + sum(sys.getsizeof(row) for row in matrix)
adj_size = sys.getsizeof(adj_list) + sum(sys.getsizeof(v) for v in adj_list.values())

print(f"Граф: {N} вузлів, {len(edges_list)} ребер")
print(f"\nМатриця суміжності:  {matrix_size:>6} байт")
print(f"Список суміжності:   {adj_size:>6} байт")
print(f"\nЕфективність матриці: {len(edges_list) * 2}/{N*N} = "
      f"{len(edges_list)*2/N**2*100:.1f}% комірок використано")
print("→ Решта 0s — чиста витрата пам'яті")

# Екстраполяція на реальну соціальну мережу
users = 1_000_000
avg_friends = 200  # Facebook: ~338 друзів, беремо 200
matrix_gb = (users ** 2) * 1 / (1024**3)  # 1 байт на комірку
adj_gb = (users + users * avg_friends) * 8 / (1024**3)  # 8 байт на pointer
print(f"\n--- Екстраполяція: {users:,} користувачів ---")
print(f"Матриця: {matrix_gb:.0f} GB")
print(f"Список:  {adj_gb:.2f} GB")

## Частина 4: Дерева vs. Графи — ієрархія проти мережі

### Усі дерева — це графи, але не всі графи — дерева

**Дерево (Tree)** — граф зі строгими обмеженнями:
- **Зв'язний** (connected): між будь-якими двома вузлами існує шлях
- **Без циклів** (acyclic): неможливо пройти по ребрах і повернутись у стартовий вузол
- **Наслідок:** $n$ вузлів → рівно $n-1$ ребер
- **Наслідок:** між будь-якими двома вузлами існує рівно **один** унікальний шлях

Дерева ідеальні для **суворих ієрархій**: файлові системи, XML, організаційні структури.

**Граф** — знімає всі ці обмеження:
- Вузол може мати кілька «батьків» (предків)
- Можливі цикли (кругові залежності, fraud rings)
- Можливі ізольовані компоненти

### Minimum Spanning Tree (MST)

З довільного зваженого графу можна видобути MST — підмножину ребер, яка:
- з'єднує **всі** вузли разом
- без циклів
- з **мінімальною сумарною вагою**

Застосування: прокладка мінімальної мережі кабелів між дата-центрами.

### Зв'язні компоненти (Connected Components)

У реальних даних граф може бути розбитий на **ізольовані острівці**.

- **Weakly Connected Component (WCC):** вузли досяжні один від одного **ігноруючи напрям** ребер.
  *Перший крок у graph data science* — перевірити, чи граф є суцільним чи розбитим.
- **Strongly Connected Component (SCC):** у спрямованому графі — вузли, між якими існує шлях **в обидва боки**, строго слідуючи напряму ребер.


## Частина 5: BFS — обхід у ширину

### Теорія

BFS досліджує граф як **хвиля, що розповсюджується від джерела**.
Алгоритм спочатку відвідує всіх **безпосередніх сусідів** стартового вузла,
потім їхніх сусідів, і так далі — рівень за рівнем.

**Ключова структура даних: FIFO черга (queue)**
- Вузли додаються у хвіст черги (`append`)
- Вузли беруться з голови черги (`popleft`)

**Математична гарантія BFS:**
> BFS вичерпно відвідує всі вузли на відстані $k$ переходів **перед** тим, як розглядати вузли на відстані $k+1$.
> Тому **перший раз**, коли BFS досягає цільового вузла — це **найкоротший шлях** (мінімум ребер).

**Складність:**
- Час: $O(|V| + |E|)$ — кожен вузол і ребро обробляється рівно один раз
- Пам'ять: $O(|V|)$ — у найгіршому випадку черга містить цілий рівень графу

**Застосування BFS:**
- Знайти найкоротший шлях (мін. переходів) у незваженому графі
- «Друзі друзів» у соціальних мережах
- Визначення найближчого таксі
- Crawling вебсторінок

### Neo4j Cypher — еквівалент BFS:

```cypher
MATCH (source:Zone {id: 132}), (target:Zone {id: 3})
MATCH p = shortestPath((source)-[:TRIP*1..10]-(target))
RETURN p
```

Neo4j виконує BFS під капотом — оптимізований двонаправлений BFS.
Замість вашого Python-коду — одна декларативна команда.


In [ ]:
# ── Частина 5: BFS — повна реалізація ──────────────────────────────────────
from collections import deque

def bfs_shortest_path(graph: dict, start: int, goal: int) -> list | None:
    """
    Знаходить найкоротший шлях (мін. переходів) між двома зонами.
    Повертає список вузлів на шляху або None якщо шлях не існує.
    """
    # Черга зберігає ШЛЯХИ, а не просто вузли
    # Кожен елемент: [поточний_вузол, список_вузлів_до_нього]
    queue = deque([[start]])

    # Множина відвіданих вузлів — не відвідувати повторно
    visited = {start}

    while queue:
        # Взяти перший шлях з черги (FIFO)
        path = queue.popleft()
        current = path[-1]  # поточний вузол = останній у шляху

        # Знайшли ціль — повернути готовий шлях
        if current == goal:
            return path

        # Обробити всіх сусідів поточного вузла
        neighbors = graph.get(current, [])
        # Якщо граф зважений [(сусід, вага), ...] — беремо тільки вузол
        if neighbors and isinstance(neighbors[0], tuple):
            neighbors = [n for n, _ in neighbors]

        for neighbor in neighbors:
            if neighbor not in visited:
                visited.add(neighbor)
                # Новий шлях = поточний шлях + новий сусід
                new_path = path + [neighbor]
                queue.append(new_path)

    return None  # цільовий вузол недосяжний


def bfs_all_distances(graph: dict, start: int) -> dict:
    """Повертає відстань (кількість хопів) від start до всіх вузлів."""
    distances = {start: 0}
    queue = deque([start])

    while queue:
        current = queue.popleft()
        neighbors = graph.get(current, [])
        if neighbors and isinstance(neighbors[0], tuple):
            neighbors = [n for n, _ in neighbors]

        for neighbor in neighbors:
            if neighbor not in distances:
                distances[neighbor] = distances[current] + 1
                queue.append(neighbor)

    return distances


# ── Тест ──────────────────────────────────────────────────────────────────
path_132_to_3 = bfs_shortest_path(taxi_graph_simple, 132, 3)
all_dist = bfs_all_distances(taxi_graph_simple, 132)

print("=== BFS Dispatch: пошук маршруту від Zone 132 до Zone 3 ===")
print(f"Найкоротший шлях: {' → '.join(map(str, path_132_to_3))}")
print(f"Кількість переходів: {len(path_132_to_3) - 1}")
print()
print("=== Відстань (хопи) від Zone 132 до всіх зон ===")
for zone, hops in sorted(all_dist.items()):
    print(f"  Zone {zone:>3}: {hops} хоп(ів)")

## Частина 6: DFS — обхід у глибину

### Теорія

DFS занурюється **якомога глибше** по одному шляху, поки не досягне глухого кута,
потім **відступає (backtrack)** і пробує інший шлях.

**Ключова структура даних: LIFO стек (stack) або рекурсія**

**Властивості DFS:**
- **Не гарантує** найкоротший шлях
- Менше використовує пам'ять, ніж BFS (не зберігає цілий рівень)
- Ефективний для пошуку циклів

**Складність:** $O(|V| + |E|)$ — та ж, що і BFS

**Застосування DFS:**
- Топологічне сортування (порядок компіляції, DAG-залежності)
- Пошук циклів (fraud rings, кругові залежності пакетів)
- Знаходження зв'язних компонент
- Генерація лабіринтів / пошук виходу

### Порівняння BFS vs. DFS

| Критерій | BFS | DFS |
|----------|-----|-----|
| Структура | FIFO черга | LIFO стек / рекурсія |
| Найкоротший шлях | ✅ Гарантує (хопи) | ❌ Не гарантує |
| Пам'ять | $O(|V|)$ — може бути великою | $O(\text{depth})$ — зазвичай менша |
| Застосування | Shortest path, broadcast | Cycle detection, topological sort |


In [ ]:
# ── Частина 6: DFS — ітеративна та рекурсивна реалізація ──────────────────

def dfs_iterative(graph: dict, start: int) -> list:
    """Ітеративний DFS — використовує явний стек."""
    visited = set()
    traversal_order = []  # порядок обходу
    stack = [start]       # LIFO стек

    while stack:
        # Беремо з вершини стеку (LIFO — останній додав, перший виймаємо)
        node = stack.pop()

        if node in visited:
            continue  # вже відвідали — пропускаємо

        visited.add(node)
        traversal_order.append(node)

        # Додати всіх НЕвідвіданих сусідів у стек
        neighbors = graph.get(node, [])
        if neighbors and isinstance(neighbors[0], tuple):
            neighbors = [n for n, _ in neighbors]

        # reverse() щоб менший сусід оброблявся першим (порядок виглядає природно)
        for neighbor in reversed(neighbors):
            if neighbor not in visited:
                stack.append(neighbor)

    return traversal_order


def dfs_find_all_paths(graph: dict, start: int, goal: int,
                        path: list = None, all_paths: list = None) -> list:
    """Рекурсивний DFS — знаходить ВСІ можливі шляхи між двома вузлами."""
    if path is None:
        path = []
    if all_paths is None:
        all_paths = []

    path = path + [start]  # не модифікуємо оригінальний список!

    if start == goal:
        all_paths.append(path)  # знайшли один шлях — зберегти
        return all_paths

    neighbors = graph.get(start, [])
    if neighbors and isinstance(neighbors[0], tuple):
        neighbors = [n for n, _ in neighbors]

    for neighbor in neighbors:
        if neighbor not in path:  # уникаємо циклів
            dfs_find_all_paths(graph, neighbor, goal, path, all_paths)

    return all_paths


# ── Тест ──────────────────────────────────────────────────────────────────
dfs_order = dfs_iterative(taxi_graph_simple, 132)
all_paths = dfs_find_all_paths(taxi_graph_simple, 132, 3)

print("=== DFS порядок обходу від Zone 132 ===")
print(" → ".join(map(str, dfs_order)))

print(f"\n=== Всі шляхи від Zone 132 до Zone 3 (DFS) ===")
for i, path in enumerate(all_paths, 1):
    print(f"  Шлях {i}: {' → '.join(map(str, path))} ({len(path)-1} переходів)")

print(f"\nBFS гарантує НАЙКОРОТШИЙ: {len(bfs_shortest_path(taxi_graph_simple, 132, 3))-1} переходів")
print(f"DFS знаходить ВСІ {len(all_paths)} можливих маршрутів")

## Частина 7: Dijkstra — найдешевший маршрут у зваженому графі

### Теорія

Dijkstra — **жадібний алгоритм** для знаходження найдешевшого маршруту
від одного вузла до всіх інших у **зваженому графі з невід'ємними вагами**.

**Ключова структура: мінімальна пріоритетна черга (min-heap)**

Замість FIFO-черги BFS, Dijkstra використовує **heap**, де кожен елемент —
`(поточна_вартість_шляху, вузол)`. Це дозволяє **завжди обирати найдешевший невідвіданий вузол**.

**Алгоритм (покроково):**
1. Ініціалізувати: `dist[start] = 0`, всі інші = `∞`
2. Додати `(0, start)` у heap
3. Поки heap не пустий:
   - Вийняти `(cost, node)` з найменшою вартістю
   - Якщо `node` вже відвіданий — пропустити
   - Для кожного сусіда: якщо `cost + edge_weight < dist[neighbor]` → **розслабити (relax)**

**Складність:** $O((|V| + |E|) \log |V|)$ — кожне ребро обробляється через heap

**Обмеження Dijkstra:** потребує **невід'ємних ваг**.
Жадібна стратегія «зафіксувати відстань при першому відвіданні» ламається
якщо пізніший шлях через від'ємне ребро дає меншу відстань.

### Neo4j Cypher — Dijkstra:

```cypher
MATCH (source:Zone {id: 132}), (target:Zone {id: 3})
CALL gds.shortestPath.dijkstra.stream('taxiGraph', {
  sourceNode: source,
  targetNode: target,
  relationshipWeightProperty: 'avg_cost'
})
YIELD path, totalCost
RETURN path, totalCost
```


In [ ]:
# ── Частина 7: алгоритм Дейкстри ──────────────────────────────────────────
import heapq

def dijkstra(graph: dict, start: int) -> tuple[dict, dict]:
    """
    Знаходить найдешевший маршрут від start до всіх вузлів.

    graph: {вузол: [(сусід, вага), ...]}
    Повертає: (distances, predecessors) для відновлення шляху.
    """
    # Початкові відстані: нескінченність для всіх, крім старту
    distances = {node: float('inf') for node in graph}
    distances[start] = 0

    # Словник попередників — для відновлення маршруту
    predecessors = {node: None for node in graph}

    # Мінімальний heap: (поточна_вартість, вузол)
    heap = [(0, start)]
    visited = set()

    while heap:
        # Взяти вузол з НАЙМЕНШОЮ поточною вартістю
        current_cost, current_node = heapq.heappop(heap)

        # Якщо вже оброблений — пропустити (можуть бути дублікати в heap)
        if current_node in visited:
            continue
        visited.add(current_node)

        # Обробити кожне ребро від поточного вузла
        for neighbor, edge_weight in graph.get(current_node, []):
            new_cost = current_cost + edge_weight

            # Relaxation: якщо новий шлях дешевший — оновити
            if new_cost < distances.get(neighbor, float('inf')):
                distances[neighbor] = new_cost
                predecessors[neighbor] = current_node
                heapq.heappush(heap, (new_cost, neighbor))

    return distances, predecessors


def reconstruct_path(predecessors: dict, start: int, goal: int) -> list:
    """Відновлює маршрут зі словника попередників."""
    path = []
    current = goal
    while current is not None:
        path.append(current)
        current = predecessors.get(current)
    path.reverse()  # зворотній порядок → прямий маршрут
    return path if path[0] == start else []


# ── Тест ──────────────────────────────────────────────────────────────────
distances, predecessors = dijkstra(taxi_graph_weighted, 132)
optimal_path = reconstruct_path(predecessors, 132, 3)

print("=== Dijkstra: найдешевший маршрут від Zone 132 ===")
print(f"{'Зона':>6}  {'Мін. вартість ($)':>18}  {'Маршрут':<30}")
print("-" * 60)
for zone, cost in sorted(distances.items()):
    path = reconstruct_path(predecessors, 132, zone)
    path_str = " → ".join(map(str, path)) if path else "недосяжно"
    print(f"Zone {zone:>3}  ${cost:>16.1f}  {path_str}")

print(f"\n✅ Найдешевший маршрут 132→3: {' → '.join(map(str, optimal_path))}")
print(f"   Загальна вартість: ${distances[3]:.1f}")

## Частина 8: Bellman-Ford — від'ємні ваги

### Коли Dijkstra не підходить?

Dijkstra **зафіксовує** відстань вузла при першому відвіданні.
Якщо граф містить **від'ємні ваги**, наступне ребро може дати меншу відстань —
жадібна стратегія Dijkstra це не врахує і дасть неправильну відповідь.

**Приклади від'ємних ваг:**
- Фінансові ринки: прибуток від конвертації валют (arbitrage)
- Логістика: знижка при певному маршруті
- Фізика: регенеративне гальмування (електромобіль)

### Bellman-Ford

Замість heap, Bellman-Ford ітеративно **релаксує всі ребра** $|V|-1$ разів:
$$\text{dist}[v] = \min(\text{dist}[v],\ \text{dist}[u] + w(u, v))$$

**Складність:** $O(|V| \times |E|)$ — набагато повільніше, ніж Dijkstra.

**Унікальна можливість Bellman-Ford:**
Якщо на $|V|$-й ітерації (після $|V|-1$ вже виконаних) відстань ще **зменшується** —
у графі є **від'ємний цикл**. Dijkstra не може виявити це взагалі.

| | Dijkstra | Bellman-Ford |
|--|----------|-------------|
| Від'ємні ваги | ❌ | ✅ |
| Виявлення від'ємних циклів | ❌ | ✅ |
| Складність | $O((V+E)\log V)$ | $O(V \times E)$ |
| Коли використовувати | Більшість задач | Від'ємні ваги, arbitrage |


## Частина 9: SQL JOIN vs. Graph Traversal

### Корінь проблеми SQL

Реляційні бази даних зберігають дані у **фіксованих таблицях**.
Зв'язки між таблицями — лише **implied** через foreign keys.

Щоб відповісти на «зв'язкове» питання, SQL виконує **JOIN** — операцію, яка
**обчислює** зв'язки у момент запиту, а не зберігає їх.

### JOIN деградує експоненціально

Знайти «друзів друзів» на глибину 4:

```sql
-- SQL: 4 JOIN-и = massive Cartesian Products
SELECT u4.name
FROM users u1
JOIN friendships f1 ON u1.id = f1.user_id
JOIN users u2       ON f1.friend_id = u2.id
JOIN friendships f2 ON u2.id = f2.user_id
JOIN users u3       ON f2.friend_id = u3.id
JOIN friendships f3 ON u3.id = f3.user_id
JOIN users u4       ON f3.friend_id = u4.id
-- ⏱️ 1500+ секунд на 1M користувачів
```

### Neo4j: та сама задача, 1 рядок Cypher

```cypher
MATCH (p:Person {name: "Alice"})-[:FRIENDS_WITH*1..4]->(friend)
RETURN friend.name
-- ⚡ ~1.3 секунди на тій самій кількості даних
```

### Чому Neo4j такий швидкий?

**Index-Free Adjacency** — ключовий архітектурний принцип:

- У SQL: знайти сусідів = **сканувати глобальний індекс** → $O(\log N)$ на кожен JOIN
- У Neo4j: знайти сусідів = **слідувати фізичному pointer-у** → $O(1)$

> **Реальний бенчмарк:**
> | Глибина пошуку | SQL | Neo4j |
> |---------------|-----|-------|
> | 2 рівні | 0.016 с | 0.001 с |
> | 3 рівні | 30.267 с | 0.003 с |
> | 4 рівні | 1543 с | 1.3 с |
> | 5 рівнів | не завершився | 2.1 с |

Продуктивність Neo4j **не залежить від загального розміру бази** —
тільки від кількості вузлів, які потрібно відвідати у конкретному запиті.


## Частина 10: Архітектура Neo4j — Index-Free Adjacency

### Neo4j — нативна граф база даних

Neo4j **не** конвертує графи у таблиці. Дані зберігаються фізично як мережа.

### Fixed-Size Records — секрет швидкості

Кожен запис вузла і ребра займає **фіксований розмір** на диску.
Це дозволяє обчислити фізичне розташування будь-якого запису
через просте множення: `offset = id × record_size` → $O(1)$.

### Node Store (9 байт на вузол)

| Поле | Розмір | Призначення |
|------|--------|-------------|
| `in_use` | 1 байт | Чи активний запис? |
| `first_rel_id` | 4 байти | Pointer на **перше ребро** вузла |
| `first_prop_id` | 4 байти | Pointer на першу властивість |

Вузол не зберігає дані — він є **навігаційним шлюзом**.

### Relationship Store (33 байти на ребро)

Кожне ребро зберігає:
- ID вузла-початку і вузла-кінця
- Тип ребра (FRIENDS_WITH, BOUGHT, etc.)
- **Попереднє і наступне ребро для обох вузлів** (doubly linked list!)

Ця doubly linked list означає, що можна обходити ребра у **будь-якому напрямку**
за $O(1)$ — запит «хто підписаний на мене?» такий само швидкий, як «на кого я підписаний?».

### Property Store

- Імена властивостей (ключі) зберігаються **один раз** у окремому файлі — дедублікація
- Малі значення (int, bool) — **inline** в запис
- Великі рядки/масиви — в окремий dynamic store

### Як це виглядає на практиці?

Запит «знайди всіх друзів Аліси»:
1. Neo4j знаходить запис Аліси → читає `first_rel_id`
2. Переходить до першого ребра → читає target node ID
3. Читає `next_rel` → переходить до наступного ребра
4. Повторює доки `next_rel == null`

**Нуль сканування індексів. Нуль Cartesian products. Чисте pointer-chasing.**


## Частина 11: Cypher — мова запитів Neo4j

### Концепція: ASCII-art синтаксис

Cypher — **декларативна** мова. Ви описуєте **патерн** у вигляді ASCII-малюнку,
і Neo4j самостійно вирішує, як його знайти.

| Синтаксис | Значення |
|-----------|----------|
| `(n)` | Вузол |
| `(n:Person)` | Вузол з міткою Person |
| `(n:Person {name: "Alice"})` | Вузол з властивістю |
| `-->` | Спрямоване ребро |
| `-[r]->` | Спрямоване ребро зі змінною r |
| `-[r:BOUGHT]->` | Ребро з конкретним типом |
| `-[*1..5]->` | Від 1 до 5 переходів |

### Основні операції

**Знайти фільми Тома Хенкса:**
```cypher
MATCH (p:Person {name: "Tom Hanks"})-[:ACTED_IN]->(m:Movie)
RETURN m.title
```

**Знайти спільних акторів:**
```cypher
MATCH (p1:Person)-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(p2:Person)
WHERE p1 <> p2
RETURN p1.name, p2.name, m.title
```

**Рекомендаційна система (2 рівні):**
```cypher
MATCH (me:User {id: "A"})-[:BOUGHT]->(p:Product)<-[:BOUGHT]-(similar:User)
MATCH (similar)-[:BOUGHT]->(rec:Product)
WHERE NOT (me)-[:BOUGHT]->(rec)
RETURN rec.name, count(similar) AS score
ORDER BY score DESC LIMIT 10
```

**Fraud detection — знайти шахраїв через спільні пристрої:**
```cypher
MATCH (fraudster:Account {known_fraud: true})-[:USES]->(device:Device)
MATCH (suspect:Account)-[:USES]->(device)
WHERE suspect <> fraudster
RETURN suspect.id, device.id
```

**Додати вузол та ребро:**
```cypher
CREATE (z:Zone {id: 132, name: "Midtown", borough: "Manhattan"})
CREATE (z2:Zone {id: 138, name: "JFK Airport"})
CREATE (z)-[:TRIP {avg_cost: 45.3, count: 1500}]->(z2)
```


## Частина 12: Реальні застосування

### 12.1 Fraud Detection & AML

Шахраї використовують **фіктивні ідентичності**, але часто переплутують спільні пристрої, телефони, IP-адреси.

SQL зламається на 5-6 JOIN при спробі простежити 5-6 рівнів. Neo4j:
- Знаходить «кільця шахрайства» за реальний час
- Виявляє Ultimate Beneficial Owner (UBO) через 10+ рівнів корпоративної власності
- Застосовує алгоритм Louvain для знаходження спільнот (fraud clusters)

### 12.2 Рекомендаційні системи

Collaborative filtering через граф:
```
User A → BOUGHT → Product X ← BOUGHT ← User B → BOUGHT → Product Y
```
Neo4j відповідає на цей запит за мілісекунди для мільйонів користувачів.

### 12.3 IT Network & Data Center Management

Граф: сервери, VM, додатки, залежності — все як вузли і ребра.

Instant impact analysis: «Якщо цей маршрутизатор відключиться — які клієнти постраждають?»
Граф обходить залежності знизу вгору за O(depth).

### 12.4 Соціальні мережі

- **PageRank**: знайти ключових influencer-ів (вузли з великим централітетом)
- **Louvain**: виявити tight-knit спільноти
- **Label Propagation**: визначити групи за інтересами

### 12.5 Біомедичні Knowledge Graphs

Вузли: хвороби, ліки, гени, симптоми.
Ребра: «лікує», «викликає», «регулює».

Граф дозволяє:
- Відкривати приховані зв'язки між хворобами
- Передбачати побічні ефекти ліків
- Repurposing: знайти нові показання для існуючих препаратів

---

## Підсумок

| Алгоритм | Структура даних | Гарантія | Складність |
|----------|----------------|----------|------------|
| BFS | FIFO черга | Найкоротший шлях (хопи) | $O(V+E)$ |
| DFS | LIFO стек / рекурсія | Всі шляхи, цикли | $O(V+E)$ |
| Dijkstra | Min-heap | Найдешевший шлях (невід'ємні ваги) | $O((V+E)\log V)$ |
| Bellman-Ford | Масив ребер | Від'ємні ваги, виявлення циклів | $O(V \times E)$ |

> **Ключовий інсайт:**
> Ми використовуємо граф-бази даних тому, що реальний світ — це густа мережа зв'язків.
> SQL **спрощує** цю мережу до плоских таблиць і **обчислює** зв'язки при кожному запиті.
> Neo4j **зберігає** мережу як вона є і **обходить** її через фізичні pointer-и за $O(1)$.
